# ¿Hasta dónde llega el reranker si le damos 200 candidatos?

**La pregunta que decide el proyecto.** Medido el 7-sep-2026 sobre las 200
consultas apartadas, el documento correcto está:

| entre los primeros | está el |
|---|---|
| 5 | 33,5% |
| 40 | ~59% |
| 100 | 70,0% |
| 200 | **75,0%** |

Es decir: **tres de cada cuatro veces ya lo tenemos recuperado y lo estamos
dejando fuera del top-5 que el modelo lee.** El reranker sobre 40 candidatos
subió el recall@5 de 33,5% a 41,0% (confirmado, p=0,0081). Esta prueba mide qué
pasa con 100 y con 200, donde hay 16 puntos más de techo.

**Por qué en GPU:** en CPU, reordenar 40 candidatos cuesta 57 s por consulta.
Doscientos costarían casi cinco minutos, y son 200 consultas: un día entero. En
una T4 esto son minutos.

**No hace falta el índice.** La búsqueda ya se hizo en el portátil; aquí solo se
puntúan pares de pregunta y pasaje. Sube `candidatos_prueba.json` a la carpeta de
Drive.

In [ ]:
import torch
assert torch.cuda.is_available(), "Sin GPU: Entorno de ejecución -> Cambiar tipo -> GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install "sentence-transformers>=3.0" 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/aliado_libre_eval'
ARCHIVO = f'{CARPETA}/candidatos_prueba.json'

import json, os
assert os.path.exists(ARCHIVO), f'No encuentro {ARCHIVO}'
datos = json.load(open(ARCHIVO, encoding='utf-8'))
print(len(datos), 'consultas |', len(datos[0]['candidatos']), 'candidatos cada una')
techo = sum(1 for d in datos if d['posicion_original']) / len(datos)
print(f'techo alcanzable: {techo*100:.1f}% (el resto no esta entre los candidatos)')

## Las ventanas a comparar

`0` es el orden que da la búsqueda hoy, sin tocar nada: es la referencia contra
la que se mide todo. El resto son cuántos candidatos ve el reranker.

In [ ]:
VENTANAS = [0, 40, 100, 200]
TOPES = [1, 5, 8]
MODELO = 'BAAI/bge-reranker-v2-m3'

from sentence_transformers import CrossEncoder
modelo = CrossEncoder(MODELO, max_length=512, device='cuda')
print('reranker listo')

In [ ]:
import time

def acierta(orden, caso, k):
    for r in orden[:k]:
        if r['id'] == caso['fragmento_id']:
            return True
        if caso.get('documento_id') and r.get('documento_id') == caso['documento_id']:
            return True
    return False

resultados = {}   # ventana -> {tope: [bool por consulta]}
tiempos = {}

for ventana in VENTANAS:
    t0 = time.time()
    aciertos = {k: [] for k in TOPES}
    for caso in datos:
        cands = caso['candidatos']
        if ventana == 0:
            orden = cands
        else:
            trozo = cands[:ventana]
            pares = [[caso['consulta'], c['texto']] for c in trozo]
            # batch_size alto porque la T4 va sobrada con 512 tokens; bajalo a 32
            # si aparece un error de memoria de CUDA.
            puntajes = modelo.predict(pares, batch_size=64, show_progress_bar=False)
            orden = [c for _, c in sorted(zip(puntajes, trozo), key=lambda x: -x[0])]
            orden = orden + cands[ventana:]
        for k in TOPES:
            aciertos[k].append(acierta(orden, caso, k))
    resultados[ventana] = aciertos
    tiempos[ventana] = (time.time() - t0) / len(datos)
    linea = '  '.join(f'@{k} {sum(aciertos[k])/len(datos)*100:5.1f}%' for k in TOPES)
    etiqueta = 'sin reranker' if ventana == 0 else f'top-{ventana}'
    print(f'{etiqueta:>13}  {tiempos[ventana]:6.2f} s/consulta   {linea}')

## ¿Es real o es ruido?

McNemar exacto sobre las mismas consultas. Mira **cuántas gana y cuántas
pierde**, no solo el porcentaje: una mejora que gana 20 y pierde 18 no es una
mejora aunque el promedio suba.

In [ ]:
from math import comb

def mcnemar(a, b):
    solo_a = sum(1 for x, y in zip(a, b) if x and not y)
    solo_b = sum(1 for x, y in zip(a, b) if y and not x)
    n = solo_a + solo_b
    if n == 0:
        return solo_a, solo_b, 1.0
    menor = min(solo_a, solo_b)
    p = sum(comb(n, i) for i in range(menor + 1)) / 2**n * 2
    return solo_a, solo_b, min(p, 1.0)

for ventana in VENTANAS[1:]:
    for k in TOPES:
        pierde, gana, p = mcnemar(resultados[0][k], resultados[ventana][k])
        veredicto = 'REAL' if p < 0.05 else 'ruido'
        print(f'top-{ventana:<3} @{k}: pierde {pierde:2}, gana {gana:2}, p={p:.4f} -> {veredicto}')

## Por perfil de usuario

El promedio esconde lo que importa: el proyecto existe para quien no sabe
escribir como un abogado. Con 25 consultas por perfil esto es una señal de
dirección, no una medición.

In [ ]:
from collections import defaultdict

mejor = max(VENTANAS[1:], key=lambda v: sum(resultados[v][5]))
por_perfil = defaultdict(lambda: {'n': 0, 'sin': 0, 'con': 0})
for i, caso in enumerate(datos):
    fila = por_perfil[caso['perfil']]
    fila['n'] += 1
    fila['sin'] += resultados[0][5][i]
    fila['con'] += resultados[mejor][5][i]

print(f"recall@5 por perfil, sin reranker vs top-{mejor}")
print()
print(f"{'perfil':26} {'sin':>6} {'con':>6}")
for perfil in sorted(por_perfil):
    f = por_perfil[perfil]
    print(f"{perfil:26} {f['sin']/f['n']*100:5.0f}% {f['con']/f['n']*100:5.0f}%")

In [ ]:
salida = {
    'modelo': MODELO,
    'n': len(datos),
    'techo': techo,
    'por_ventana': {
        str(v): {'segundos': tiempos[v], **{str(k): sum(resultados[v][k]) for k in TOPES}}
        for v in VENTANAS
    },
}
ruta = f'{CARPETA}/reranker_profundo.json'
json.dump(salida, open(ruta, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('guardado en', ruta)

## Cómo leer el resultado

- **Si top-200 acerca el @5 al 60-70%**: el reranker profundo es el camino, y la
  conversación pasa a ser cuánto cuesta una GPU en el servidor.
- **Si top-200 apenas mejora sobre top-40**: el reranker ya dio lo que tenía, y
  el esfuerzo se va a afinar el embedding (`colab_afinar_embedding.ipynb`), que
  ataca el orden desde el otro lado.
- **Si el tiempo por consulta en GPU sigue siendo de segundos**: eso decide la
  arquitectura tanto como el acierto. Una búsqueda que tarda cinco segundos no la
  usa nadie, por buena que sea.

Pase lo que pase, el techo de esta prueba es el 75%: lo que no está entre los
candidatos no lo arregla ningún reordenamiento. Ese 25% restante es otro
problema, y es el siguiente.